# EEG Preprocessing Pipeline Tutorial

## Part of the open_dvm Toolbox

This tutorial demonstrates how to preprocess raw EEG data using the open_dvm toolbox. We cover data loading, artifact correction, filtering, and ICA-based component removal to prepare data for subsequent ERP, TFR, BDM, and CTF analyses.

## Learning Objectives

After completing this tutorial, you will:

- **Understand the open_dvm preprocessing workflow** — Load raw EEG and behavioral data
- **Configure preprocessing parameters** — Set filtering, ICA, and artifact rejection settings
- **Handle subject-specific issues** — Define bad channels per subject
- **Integrate eye-tracking data** — Exclude saccades and fixation breaks
- **Process behavioral metadata** — Align behavioral events with EEG data
- **Verify preprocessed data** — Check data quality and trial counts
- **Save processed epochs** — Create publication-ready datasets for analysis

**Prerequisites:** Raw EEG data and behavioral logs from your experiment. This tutorial uses example data from a visual search task with 7 subjects.

## Overview: The Preprocessing Pipeline

### Data Flow

Raw EEG → Load & Montage → Filter → Epoch → Link Behavior/Eye Data → ICA → Autoreject → Save

### Key Steps

1. **Load raw EEG** — Import continuous EEG data from BioSemi or other formats
2. **Apply montage** — Define electrode positions (e.g., BioSemi-32)
3. **Filter data** — High-pass (remove drift) and notch (remove line noise + harmonics)
4. **Create epochs** — Segment continuous data around trigger events
5. **Update metadata** — Link behavioral variables and eye-tracking data to each epoch
6. **Run ICA** — Independent Component Analysis to identify and remove artifacts
7. **Apply autoreject** — Automatic bad segment detection and interpolation
8. **Save epochs** — Store preprocessed data for analysis

Eye-tracking *exclusion* (dropping trials with saccades or fixation breaks) happens later, when loading preprocessed data for analysis (Section 4) -- not during preprocessing itself.

## Section 1: Setup and Configuration

### 1.1 Import Required Libraries

In [ ]:
import os

# Import open_dvm modules
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis.preprocessing_pipeline import eeg_preprocessing_pipeline
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.support.preprocessing_utils import format_subject_id

print("✓ open_dvm preprocessing module imported successfully!")

### 1.2 Define Project Structure and Paths

Set the root project folder and initialize the FolderStructure tracker. This defines where raw data, preprocessed data, and behavioral files are stored.

In [ ]:
# Download (if not already cached) and locate the raw tutorial dataset
from open_dvm.support.datasets import fetch_raw_data

project_folder = fetch_raw_data()
os.chdir(project_folder)

# Initialize FolderStructure to manage file organization
fs = FolderStructure()

print(f'✓ Project folder set to: {project_folder}')
print(f'✓ FolderStructure initialized')

### 1.3 Define Subject-Specific Information

Specify any known bad channels for each subject (e.g., from electrode contact issues). These channels will be interpolated during preprocessing.

In [3]:
# Subject-specific information (bad channels identified during recording)
sj_info = {
    '1': {'bad_chs': []},              # Subject 1: no bad channels
    '2': {'bad_chs': ['C4', 'CP2']},   # Subject 2: C4 and CP2 had contact issues
    '3': {'bad_chs': ['C4', 'CP2']},
    '4': {'bad_chs': ['P3', 'P7']},
    '5': {'bad_chs': []},
    '6': {'bad_chs': []},
    '7': {'bad_chs': []}
}

print('Subject-specific bad channels:')
for sj, info in sj_info.items():
    if info['bad_chs']:
        print(f'  Subject {sj}: {info["bad_chs"]}')
    else:
        print(f'  Subject {sj}: none')

Subject-specific bad channels:
  Subject 1: none
  Subject 2: ['C4', 'CP2']
  Subject 3: ['C4', 'CP2']
  Subject 4: ['P3', 'P7']
  Subject 5: none
  Subject 6: none
  Subject 7: none


## Section 2: Preprocessing Parameters

### 2.1 EEG Recording Parameters

Define the characteristics of the EEG recording (sampling rate, number of sessions, electrode montage, etc.).

In [ ]:
# EEG recording setup
montage = 'biosemi32'              # Electrode montage
nr_sessions = 1                    # Number of independent EEG sessions
nr_sjs = 7                         # Number of subjects
eeg_runs = [1]                     # Recording runs within session (list for multiple runs)

# Channel information
eog = ['V_up', 'V_do', 'H_r', 'H_l']  # EOG channels (electrode-based, not the eye tracker): vertical up/down, horizontal right/left
ref = ['Ref_r', 'Ref_l']                # Reference channels (BioSemi separate references)

print(f'Recording configuration:')
print(f'  Montage: {montage}')
print(f'  Sessions: {nr_sessions}')
print(f'  Subjects: {nr_sjs}')
print(f'  Runs per session: {eeg_runs}')
print(f'  EOG channels: {eog}')
print(f'  Reference channels: {ref}')

### 2.2 Epoching Parameters

Define how to extract epochs from continuous EEG data around trigger events.

In [5]:
# Epoching setup
trigger_header = 'display_trigger'  # Behavioral column containing trigger codes
event_id = list(range(1, 250))      # Trigger codes to epoch (1-249)
t_min = -0.2                        # Time before trigger (seconds)
t_max = 0.5                         # Time after trigger (seconds)
flt_pad = 0.5                       # Epoch extension to control for filter artifacts

print(f'Epoching configuration:')
print(f'  Trigger header: {trigger_header}')
print(f'  Event codes: {len(event_id)} triggers ({min(event_id)}-{max(event_id)})')
print(f'  Epoch window: [{t_min:.1f}, {t_max:.1f}] seconds')
print(f'  Filter padding: {flt_pad} seconds')

Epoching configuration:
  Trigger header: display_trigger
  Event codes: 249 triggers (1-249)
  Epoch window: [-0.2, 0.5] seconds
  Filter padding: 0.5 seconds


### 2.3 Filtering and Artifact Correction Parameters

Specify high-pass filtering, notch filtering, ICA, and autoreject settings.

In [ ]:
# Preprocessing parameter dictionary
preproc_param = {
    'high_pass': 0.01,      # High-pass filter cutoff (Hz) - removes DC drift
    'run_ica': True,        # Run Independent Component Analysis
    'run_autoreject': True, # Run automatic artifact rejection
    'notch': True           # Apply notch filter at 50/100/150 Hz (line noise + harmonics)
}

print('Preprocessing parameters:')
for key, value in preproc_param.items():
    print(f'  {key}: {value}')

### 2.4 Eye-Tracking Quality Control

Define parameters for excluding trials with eye movement artifacts (saccades, fixation breaks).

In [ ]:
# Eye-tracking information and quality control
eye_info = {
    'eog': eog,  # EOG channel names: [vertical up, vertical down, horizontal right, horizontal left]
    'tracker_ext': 'asc',           # File extension for eye tracker data
    'sfreq': 1000,                  # Eye tracker sampling rate (Hz)
    'trigger_msg': 'Onset search',  # In-trial event that window_oi is aligned to (e.g. stimulus onset)
    'window_oi': (-700, 1000),      # Time window around trigger_msg (ms)
    'start': 'start trial',         # Marker segmenting the eye-tracker recording into trials
    'drift_correct': None,          # Time window (ms) for gaze drift correction, or None to skip
    'viewing_dist': 70,             # Viewing distance (cm)
    'screen_res': (1920, 1080),     # Screen resolution (pixels)
    'screen_h': 29                  # Screen height (cm)
}

print('Eye-tracking information:')
for key, value in eye_info.items():
    print(f'  {key}: {value}')

## Section 3: Run the Preprocessing Pipeline

### 3.1 Preprocess a Single Subject

Run the complete preprocessing pipeline for one subject as an example. This step takes ~5-10 minutes per subject depending on data length and computational resources.

In [ ]:
# Preprocess subject 1 (example)
sj = 1

print(f'Starting preprocessing for subject {sj}...')
print('This may take several minutes...\n')

# Call the preprocessing pipeline
eeg_preprocessing_pipeline(
    sj=sj,                              # subject ID to preprocess
    session=1,                          # EEG session number
    eog=eog,                            # EOG channels (Section 2.1)
    ref=ref,                            # reference channels (Section 2.1)
    eeg_runs=eeg_runs,                  # recording runs within the session (Section 2.1)
    nr_sessions=nr_sessions,            # number of independent EEG sessions (Section 2.1)
    t_min=t_min,                        # epoch start relative to trigger, in seconds (Section 2.2)
    t_max=t_max,                        # epoch end relative to trigger, in seconds (Section 2.2)
    flt_pad=flt_pad,                    # epoch padding to absorb filter edge artifacts (Section 2.2)
    sj_info=sj_info,                    # per-subject bad-channel info (Section 1.3)
    eye_info=eye_info,                  # eye-tracking channels and QC settings (Section 2.4)
    event_id=event_id,                  # trigger codes to epoch (Section 2.2)
    montage=montage,                    # electrode montage (Section 2.1)
    preproc_param=preproc_param,        # filtering/ICA/autoreject settings (Section 2.3)
    trigger_header=trigger_header,      # behavioral column holding trigger codes (Section 2.2)
    beh_oi=None,                        # select all columns in behavioral data (set to None)
    binary=0,                           # no stim-channel offset correction needed for this dataset
    preproc_name='main',                # label used when saving preprocessed output files
    nr_sjs=nr_sjs,                      # total number of subjects (Section 2.1)
    excl_factor=None                    # no trial-level exclusion at this stage
)

print(f'✓ Subject {sj} preprocessing complete!')

### 3.2 Batch Process All Subjects

Run preprocessing for all subjects in the dataset. Uncomment and modify below to process multiple subjects.

In [ ]:
#  process all subjects 
separator = '=' * 60
for sj in range(2, 8):  # Subjects 2-7
    print(f'\n{separator}')
    print(f'Processing subject {sj}...')
    print(separator)
    
    eeg_preprocessing_pipeline(
        sj=sj,                              # subject ID to preprocess
        session=1,                          # EEG session number
        eog=eog,                            # EOG channels (Section 2.1)
        ref=ref,                            # reference channels (Section 2.1)
        eeg_runs=eeg_runs,                  # recording runs within the session (Section 2.1)
        nr_sessions=nr_sessions,            # number of independent EEG sessions (Section 2.1)
        t_min=t_min,                        # epoch start relative to trigger, in seconds (Section 2.2)
        t_max=t_max,                        # epoch end relative to trigger, in seconds (Section 2.2)
        flt_pad=flt_pad,                    # epoch padding to absorb filter edge artifacts (Section 2.2)
        sj_info=sj_info,                    # per-subject bad-channel info (Section 1.3)
        eye_info=eye_info,                  # eye-tracking channels and QC settings (Section 2.4)
        event_id=event_id,                  # trigger codes to epoch (Section 2.2)
        montage=montage,                    # electrode montage (Section 2.1)
        preproc_param=preproc_param,        # filtering/ICA/autoreject settings (Section 2.3)
        trigger_header=trigger_header,      # behavioral column holding trigger codes (Section 2.2)
        beh_oi=None,                        # select all columns in behavioral data (set to None)
        binary=0,                           # no stim-channel offset correction needed for this dataset
        preproc_name='main',                # label used when saving preprocessed output files
        nr_sjs=nr_sjs,                      # total number of subjects (Section 2.1)
        excl_factor=None                    # no trial-level exclusion at this stage
    )
    
    print(f'✓ Subject {sj} complete')

print(f'\n{separator}')
print('✓ All subjects preprocessing complete!')
print(separator)

## Section 4: Verify Preprocessed Data

### 4.1 Load and Inspect Preprocessed Epochs

In [ ]:
# Load the preprocessed data for inspection
sj = 1

eye_dict = {
    'use_tracker': True,
    'window_oi': (0, 0.3),
    'angle_thresh': 1,
    'viewing_dist': 70,
    'screen_res': (1920, 1080),
    'screen_h': 29,
    'drift_correct': (-0.2, 0)
}

df, epochs = FolderStructure().load_processed_epochs(
    sj=sj,                       # subject ID
    fname='ses_01_main',         # preprocessed file name (sub_<sj>_ses_01_main-epo.fif)
    preproc_name='main',         # preprocessing pipeline name (locates parameter files)
    eye_dict=eye_dict            # eye-tracking exclusion criteria
)

print(f'✓ Data loaded for subject {sj}')
print(f'\nEpochs: {len(epochs)} trials')
print(f'Channels: {len(epochs.ch_names)}')
print(f'Sampling rate: {epochs.info["sfreq"]} Hz')
print(f'Time range: {epochs.tmin:.3f} to {epochs.tmax:.3f} s')
print(f'\nBehavioral data shape: {df.shape[0]} trials × {df.shape[1]} variables')
print(f'\nBehavioral variables:')
print(df.columns.tolist())

### 4.2 Check Trial Counts by Condition

In [11]:
# Check trial counts across conditions
print('Trial counts by experimental conditions:\n')

print('By block type:')
print(df['block_type'].value_counts())

print('\nBy target condition:')
print(df['target_cnd'].value_counts())

print('\nBy distractor condition:')
print(df['dist_cnd'].value_counts())

print('\nCross-condition:')
print(df[['target_cnd', 'dist_cnd']].value_counts().head(10))

Trial counts by experimental conditions:

By block type:
block_type
main         1900
localizer     953
Name: count, dtype: int64

By target condition:
target_cnd
train      953
absent     953
present    947
Name: count, dtype: int64

By distractor condition:
dist_cnd
absent     1905
present     948
Name: count, dtype: int64

Cross-condition:
target_cnd  dist_cnd
train       absent      953
absent      absent      478
            present     475
present     absent      474
            present     473
Name: count, dtype: int64


## Section 5: Summary and Next Steps

### What You've Accomplished

1. ✓ Set up the project folder and configuration
2. ✓ Defined subject-specific preprocessing parameters
3. ✓ Configured filtering, ICA, and artifact rejection
4. ✓ Integrated eye-tracking quality control
5. ✓ Ran the complete preprocessing pipeline
6. ✓ Verified the preprocessed epochs

### Output Files

The preprocessing pipeline saves:
- **Preprocessed epochs**: `eeg/processed/sub_*_ses_1_main-epo.fif` (behavioral data is attached as `epochs.metadata`, not a separate file)
- **Quality reports**: `preprocessing/report/main/sj_*_ses_*.html`
- **Group-level log**: `preprocessing/group_info/preproc_param_main.json` (per-subject trial counts, interpolated channels, etc.)

### Next Steps

Now that you have preprocessed EEG data, you can proceed to:

1. **ERP Analysis** → See `02_erp_analysis.ipynb` for computing event-related potentials
2. **TFR Analysis** → Time-frequency analysis of oscillatory activity
3. **BDM Analysis** → Multivariate classification/decoding analysis
4. **CTF Analysis** → Spatial channel tuning functions (inverted encoding model)

All subsequent analyses use the preprocessed epochs loaded via `FolderStructure().load_processed_epochs()`.